# Training a Logistic Regression Classification Model

Logistic Regression is one of the most important classification models in machine learning. It predicts the probability of class membership and gives you an interpretable benchmark before moving to more complex methods.

This lesson covers how Logistic Regression works, how to evaluate it correctly, how to interpret coefficients as odds ratios, and how to keep preprocessing leakage-free.

## Why Logistic Regression matters

Use Logistic Regression when you need a strong, fast, interpretable classification model. It is especially useful as the first real model after a majority-class baseline.

It is often a good choice when you want:

- Probabilistic predictions
- Clear coefficient interpretation
- A reliable benchmark for more complex models
- A pipeline-friendly classification workflow

In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(79)
n_rows = 600

df = pd.DataFrame({
    'tenure': rng.integers(1, 72, size=n_rows),
    'MonthlyCharges': rng.normal(70, 15, size=n_rows).round(2),
    'TotalCharges': rng.normal(2500, 800, size=n_rows).round(2),
})

logit = -0.03 * df['tenure'] + 0.02 * df['MonthlyCharges'] + 0.0004 * df['TotalCharges']
prob = 1 / (1 + np.exp(-logit))
df['Churn'] = (rng.random(n_rows) < prob).astype(int)

df.head()

## Baseline and model

Always split before fitting anything. The baseline should be a majority-class predictor on the training split, and the Logistic Regression model should be evaluated on the same held-out test set.

In [ ]:
TARGET_COLUMN = 'Churn'
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

pipeline.fit(X_train, y_train)
model_pred = pipeline.predict(X_test)
model_prob = pipeline.predict_proba(X_test)[:, 1]

def evaluate_classifier(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    print(f'{name:25s} | Accuracy: {acc:.3f} | ROC-AUC: {auc:.3f}')

evaluate_classifier('Baseline (majority)', y_test, baseline_pred, baseline_prob)
evaluate_classifier('Logistic Regression', y_test, model_pred, model_prob)

## Classification report and coefficients

Accuracy alone is not enough. You should inspect precision, recall, and F1, and then look at coefficients as odds-ratio signals when the features are scaled.

In [ ]:
print('--- Logistic Regression Report ---')
print(classification_report(y_test, model_pred))

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': pipeline.named_steps['model'].coef_[0],
    'Odds Ratio': np.exp(pipeline.named_steps['model'].coef_[0]),
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"Intercept: {pipeline.named_steps['model'].intercept_[0]:.3f}")
print(coef_df.to_string(index=False))

## Cross-validation and practical checklist

Cross-validation helps confirm that the model is stable, not just lucky on one split.

Checklist:

- Use a stratified train/test split.
- Fit scaling only on the training data through a pipeline.
- Compare against a majority-class baseline on accuracy and ROC-AUC.
- Inspect the full classification report, not just one metric.
- Check cross-validation mean and standard deviation.

## Bonus resources

- Scikit-learn LogisticRegression Documentation
- Scikit-learn Linear Models - Logistic Regression

In [ ]:
cv_auc = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc')
cv_f1 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')

print(f'CV ROC-AUC: {cv_auc.round(3)}')
print(f'Mean AUC:   {cv_auc.mean():.3f} +/- {cv_auc.std():.3f}')
print(f'\nCV F1:      {cv_f1.round(3)}')
print(f'Mean F1:    {cv_f1.mean():.3f} +/- {cv_f1.std():.3f}')